## 1 · Setup and Folder Check
We identify the project root automatically so this works no matter where you run it (VS Code, Jupyter, etc.).
We then verify that all required folders and raw data files exist.

In [29]:
import os
from pathlib import Path

def find_repo_root(start: Path = Path.cwd()):
    for p in [start, *start.parents]:
        if (p / "data").exists():
            return p
    return start

ROOT = find_repo_root()
os.chdir(ROOT)
print("Project Root :", ROOT)
print("Current Dir :", Path.cwd())

DATA_RAW_REDDIT  = ROOT / "data" / "raw" / "reddit"
DATA_RAW_TWITTER = ROOT / "data" / "raw" / "Mental-Health-Twitter.csv"

print("Reddit Exists :", DATA_RAW_REDDIT.exists())
print("Twitter File Exists :", DATA_RAW_TWITTER.exists())


Project Root : c:\Users\halab_iqyec8m\crossplatform-mentalhealth-ml
Current Dir : c:\Users\halab_iqyec8m\crossplatform-mentalhealth-ml
Reddit Exists : True
Twitter File Exists : True


2 · Imports and Global Configuration

In [30]:
import re
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.utils import shuffle
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

SEED = 42
np.random.seed(SEED)

OUT_PREP_REDDIT   = ROOT / "data" / "preprocessed" / "reddit"
OUT_PREP_TWITTER  = ROOT / "data" / "preprocessed" / "twitter"
OUT_PREP_VECTORS  = ROOT / "data" / "preprocessed" / "vectorizers"

for d in [OUT_PREP_REDDIT, OUT_PREP_TWITTER, OUT_PREP_VECTORS]:
    d.mkdir(parents=True, exist_ok=True)


## 3 · Text Cleaning Functions
These remove URLs, punctuation, digits, and normalize whitespace.
They also lowercase everything for consistent tokenization.

In [31]:
def clean_for_rf(text: str) -> str:
    if not isinstance(text, str):
        text = str(text)
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"[^A-Za-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip().lower()
    return text


## 4 · Reddit Dataset Loader
We gather all 12 subreddits (6 mental-health, 6 control)
and label them as binary 1 vs 0.
Then we clean and shuffle the combined DataFrame.

In [32]:
def load_reddit_dataset():
    MH_SUBS   = ["addiction", "adhd", "alcoholism", "anxiety", "mentalhealth", "suicidewatch"]
    CTRL_SUBS = ["conspiracy", "fitness", "jokes", "meditation", "relationships", "teaching"]

    rows = []
    for sub, label in [(MH_SUBS, 1), (CTRL_SUBS, 0)]:
        for s in sub:
            matches = list(DATA_RAW_REDDIT.glob(f"*{s}*.csv"))
            if not matches:
                print(f"⚠️ Missing file for {s}")
                continue
            df = pd.read_csv(matches[0])
            if "post" not in df.columns:
                print(f"⚠️ 'post' column missing in {matches[0].name}")
                continue
            df = df[["post"]].rename(columns={"post": "text"})
            df["label"] = label
            df["source"] = s
            rows.append(df)

    combined = pd.concat(rows, ignore_index=True)
    combined["cleaned"] = combined["text"].map(clean_for_rf)
    combined = shuffle(combined, random_state=SEED).reset_index(drop=True)
    return combined

reddit_df = load_reddit_dataset()
print("Shape :", reddit_df.shape)
reddit_df.head(3)


Shape : (124444, 4)


,text,label,source,cleaned
0,The mind keeps going back for more punishment ...,1,anxiety,the mind keeps going back for more punishment ...
1,I (M24) don't think I love my girlfriend (F24)...,0,relationships,i m don t think i love my girlfriend f anymore...
2,My (23M) Boyfriend is constantly accidentally ...,0,relationships,my m boyfriend is constantly accidentally maki...


## 5 · Train/Validation/Test Split
70 % train · 15 % validation · 15 % test
Stratified by label to maintain class balance.

In [33]:
train_df, temp_df = train_test_split(reddit_df, test_size=0.30, stratify=reddit_df["label"], random_state=SEED)
val_df, test_df   = train_test_split(temp_df, test_size=0.50, stratify=temp_df["label"], random_state=SEED)

for name, d in [("Train",train_df),("Val",val_df),("Test",test_df)]:
    print(f"{name}: {len(d)} samples → {d['label'].value_counts().to_dict()}")


Train: 87110 samples → {0: 52278, 1: 34832}
Val: 18667 samples → {0: 11203, 1: 7464}
Test: 18667 samples → {0: 11203, 1: 7464}


## 6 · TF-IDF Feature Extraction for Random Forest
Random Forest does not need scaling but benefits from
compact vocabularies and n-grams.
We cap the vocabulary to 3000 terms and save the fitted vectorizer.

In [34]:
vectorizer = TfidfVectorizer(
    max_features=3000,
    min_df=3,
    max_df=0.9,
    ngram_range=(1,2),
    sublinear_tf=True,
    stop_words="english"
)

X_train = vectorizer.fit_transform(train_df["cleaned"])
X_val   = vectorizer.transform(val_df["cleaned"])
X_test  = vectorizer.transform(test_df["cleaned"])

joblib.dump(vectorizer, OUT_PREP_VECTORS / "tfidf_rf.pkl")
print("Vectorizer saved →", OUT_PREP_VECTORS / "tfidf_rf.pkl")


Vectorizer saved → c:\Users\halab_iqyec8m\crossplatform-mentalhealth-ml\data\preprocessed\vectorizers\tfidf_rf.pkl


## 7 · Save Preprocessed Reddit Files
Each split is saved separately for easy reuse in model training.

In [35]:
pd.DataFrame(X_train.toarray()).assign(label=train_df["label"].values)\
    .to_csv(OUT_PREP_REDDIT/"reddit_rf_train.csv", index=False)
pd.DataFrame(X_val.toarray()).assign(label=val_df["label"].values)\
    .to_csv(OUT_PREP_REDDIT/"reddit_rf_val.csv", index=False)
pd.DataFrame(X_test.toarray()).assign(label=test_df["label"].values)\
    .to_csv(OUT_PREP_REDDIT/"reddit_rf_test.csv", index=False)

print("Reddit preprocessed data saved in :", OUT_PREP_REDDIT)


Reddit preprocessed data saved in : c:\Users\halab_iqyec8m\crossplatform-mentalhealth-ml\data\preprocessed\reddit


## 8 · Twitter Dataset Processing
We clean text in the same way and
reuse the Reddit TF-IDF vectorizer
to guarantee a shared feature space for cross-platform evaluation.

In [36]:
tw = pd.read_csv(DATA_RAW_TWITTER)

# Detect text and label columns
text_col  = "post_text" if "post_text" in tw.columns else ("tweet" if "tweet" in tw.columns else None)
label_col = "label" if "label" in tw.columns else None
if text_col is None or label_col is None:
    raise ValueError(f"Columns found: {tw.columns.tolist()} — expected post_text/tweet and label")

tw["cleaned"] = tw[text_col].map(clean_for_rf)

vectorizer = joblib.load(OUT_PREP_VECTORS / "tfidf_rf.pkl")
X_twitter  = vectorizer.transform(tw["cleaned"])

pd.DataFrame(X_twitter.toarray()).assign(label=tw[label_col].values)\
    .to_csv(OUT_PREP_TWITTER/"twitter_rf.csv", index=False)

print("Twitter data processed and saved to :", OUT_PREP_TWITTER)


Twitter data processed and saved to : c:\Users\halab_iqyec8m\crossplatform-mentalhealth-ml\data\preprocessed\twitter


## 9 · Quick Sanity Check / Mini Evaluation
Optional: fit a small Random Forest on Reddit train and validate on Reddit val
to verify everything works as expected.

In [37]:
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    n_jobs=-1,
    random_state=SEED
)
rf.fit(X_train, train_df["label"])
pred_val = rf.predict(X_val)
print(classification_report(val_df["label"], pred_val))


              precision    recall  f1-score   support

           0       0.93      0.95      0.94     11203
           1       0.93      0.89      0.91      7464

    accuracy                           0.93     18667
   macro avg       0.93      0.92      0.93     18667
weighted avg       0.93      0.93      0.93     18667



## 10 · Expected Outputs

- data/preprocessed/reddit/reddit_rf_train.csv

- data/preprocessed/reddit/reddit_rf_val.csv

- data/preprocessed/reddit/reddit_rf_test.csv

- data/preprocessed/twitter/twitter_rf.csv

- data/preprocessed/vectorizers/tfidf_rf.pkl

## Reference Notes

- "Parameters tuned according to Pedregosa et al., Journal of Machine Learning Research 12 (2011), 2825–2830."

- "TF-IDF weighting follows Ramos (2003) “Using TF-IDF to Determine Word Relevance in Document Queries.”

- "Random Forest setup mirrors standard scikit-learn guidelines (max_features≈√p)."